#### __libraries__

In [146]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

In [148]:
from shared_scripts.statistical_analysis import t_margin_of_error

In [150]:
from shared_scripts.bb2_data_load import BB2_load_into_dataframe

#### __plt params__

In [560]:
plt.rcParams["legend.frameon"] = False
plt.rcParams['legend.fontsize'] = 12
plt.rcParams['axes.labelsize'] = 19
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['xtick.labelsize'] = 16
plt.rcParams['ytick.labelsize'] = 16  
plt.rcParams['font.size'] = 12
plt.rcParams['figure.dpi'] = 1000
plt.rcParams['font.family'] = 'Calibri'

#### __func__

In [568]:
def _2B_data_process(data, **kwargs):
    
    #chck the input data
    if not isinstance(data, pd.DataFrame) or data.empty:
        raise ValueError("input must be a non-empty pandas DataFrame")
        
    if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in data.dtypes):
        raise TypeError("All columns in inout must be numerical")
        
    #assign timepoint
    frq= kwargs.get('frequency', np.NaN)
    indel= kwargs.get('initial_delay', np.NaN)
    if "frequency" not in kwargs or "initial_delay" not in kwargs:
        raise ValueError("frequency and initial_delay must be provided")
    try:
        data= data.assign(timepoint_minutes=data['frame'] * frq + indel).drop(columns=["frame"])
    except Exception as ex:
        raise ValueError("failed to calculate the timepoints in minutes")
        
    #set the index and rename the remaining columns
    if 'timepoint_minutes' in data.columns:
        data= data.set_index('timepoint_minutes')
    else:
        raise ValueError("dataset is missing a frame column")
        
    if len(data.columns)== 3:
        data.columns= ['rep1', 'rep2', 'rep3']
    else:
        raise ValueError("dataset is missing a frame column")
        
    #calculate the mean
    try:
        data= data.assign(avg_distance= data.mean(axis= 1))
    except Exception as ex:
        print(f'failed to calculate the mean: {ex}')
        data= data.assign(avg_distance= np.NaN)
        
    #calculate the MOE95
    try:
        data= data.assign(moe95= data.apply(lambda x: t_margin_of_error(x.iloc[0:3]), axis= 1))
    except Exception as ex:
        print(f'failed to calculate the moe95: {ex}')
        data= data.assign(moe95= np.NaN)
        
    return data


def fig2(_2A_data, _2B_data, **kwargs):
    
    #fig + grid
    fig= plt.figure(figsize= (12.8, 5), constrained_layout= True)    
    gs= gridspec.GridSpec(1,4, figure=fig)
    
    #gridspec
    ax1= fig.add_subplot(gs[0:1, 0:1])
    ax2= fig.add_subplot(gs[0:1, 1:2])    
    ax3= fig.add_subplot(gs[0:1, 2:4])    
    
    ##2A
    #alpha
    box_position = 1.5
    # Plot boxplot
    ax1.boxplot(_2A_data.alpha, 
                positions=[box_position], 
                widths=0.5,
                showfliers=False,  
                patch_artist= True,
                boxprops=dict(facecolor="#DFE9F5", edgecolor="#00527C", linewidth=2),
                whiskerprops=dict(color="#00527C", linewidth=2),
                capprops=dict(color="#00527C", linewidth=2),
                medianprops=dict(color="#00527C", linewidth=2.5))

    x_points = np.random.uniform(0.25, 0.75, len(_2A_data.alpha))
    ax1.scatter(x_points,
                _2A_data.alpha,
                alpha=0.66,
                color="#DFE9F5",
                edgecolors="white",
                linewidths=0.2,
                s=12,
                zorder=3)
    ax1.set_xticks([])
    ax1.set_ylabel('alpha', weight= 'bold')
    ax1.set_ylim(bottom=-0.55, top= 2.55)
    ax1.set_yticks([-0.5, 0.5, 1.5, 2.5])
    ax1.axhline(_2A_data.alpha.median(), ls= '--', lw= .66, color="#00527C", alpha= .66, zorder= 5)
    ax1.set_xlim(0, 2)
    
    #K
    ax2.boxplot(_2A_data.K, 
                positions=[box_position], 
                widths=0.5,
                showfliers=False,  
                patch_artist= True,
                boxprops=dict(facecolor="#DFE9F5", edgecolor="#00527C", linewidth=2),
                whiskerprops=dict(color="#00527C", linewidth=2),
                capprops=dict(color="#00527C", linewidth=2),
                medianprops=dict(color="#00527C", linewidth=2.5))

    x_points = np.random.uniform(0.25, 0.75, len(_2A_data.K))
    ax2.scatter(x_points,
                _2A_data.K,
                alpha=0.66,
                color="#DFE9F5",
                edgecolors="white",
                linewidths=0.2,
                s=12,
                zorder=3)
    ax2.set_xticks([])
    ax2.set_ylabel('K', weight= 'bold')
    ax2.set_ylim(bottom=-2, top= 92)
    ax2.set_yticks([0, 30, 60, 90])
    ax2.axhline(_2A_data.K.median(), ls= '--', lw= .66, color="#00527C", alpha= .66, zorder= 5)
    ax2.set_xlim(0, 2)
    
    ##2B
    ax3.errorbar(_2B_data.index,
                 _2B_data.avg_distance,
                 lw= 2.5,
                 label= 'control',
                 color= '#00527C')
    y_min_ctrl= _2B_data.avg_distance - _2B_data.moe95
    y_max_ctrl= _2B_data.avg_distance + _2B_data.moe95
    ax3.fill_between(_2B_data.index, 
                     y_min_ctrl, 
                     y_max_ctrl, 
                     color='#DFE9F5', 
                     label="margin of error area: control")
    ax3.set_xlabel('time (min)', weight= 'bold')
    ax3.set_ylabel('average movement', weight= 'bold') #pxls/frame
    ax3.set_ylim(1.5, 6)
    ax3.set_yticks([1.5, 3, 4.5, 6])
    ax3.set_xticks([0,60,120,180,240,300,360,420,480,540])
    
    ##export
    if kwargs.get('export', False):
        from pathlib import Path
        if not kwargs.get('path'):
            raise ValueError("export path must be provided.")
        output_path = Path(kwargs.get('path'))
        if not output_path.parent.exists():
            raise FileNotFoundError(f"directory does not exist: {output_path}")
        try:
            plt.savefig(output_path, dpi= 1000)
        except Exception as ex:
            raise OSError(f"Failed to export plot to '{kwargs.get('path')}': {ex}")

#### __data Load__

In [562]:
_2A= BB2_load_into_dataframe('Figure_2A_full_dataset.csv')

In [564]:
_2B= BB2_load_into_dataframe('Figure_2B.csv')

#### __data processing__

In [566]:
_2B_final= _2B_data_process(_2B, frequency= 3.5, initial_delay= 7)

#### __visuals__

In [570]:
fig2(_2A,
     _2B_final,
     export= False,
     path= r"...\Figure2.png")